# Figure 3

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from plot_utils import check_data, load_integrated_table, table
check_data()


## Panel A

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from IPython.display import Image, display

_KNOWN_ROOT = Path.cwd() / "04_plotting_workspace" / "nature_story_atlas_complete_clean_20260702"
_CANDIDATES = [Path.cwd(), _KNOWN_ROOT]
ROOT = None
for _cand in _CANDIDATES:
    if (_cand / "data/nature_story_atlas.feather").exists():
        ROOT = _cand
        break
if ROOT is None:
    raise FileNotFoundError("nature_story_atlas.feather was not found. Open this notebook from its plotting folder or update _KNOWN_ROOT.")

DATA = ROOT / "data/nature_story_atlas.feather"
OUT = ROOT / "figures" / "panels" / "main_figure3_a_split_power_rings_candidate.png"
OUT.parent.mkdir(parents=True, exist_ok=True)

mpl.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans", "sans-serif"],
    "font.size": 8,
    "axes.titlesize": 9,
    "axes.labelsize": 8,
    "xtick.labelsize": 7,
    "ytick.labelsize": 7,
    "legend.fontsize": 7,
    "axes.linewidth": 0.8,
})

FAMILY_COLOR = {
    "local_static": "#C85A2E",
    "local_temporal": "#E49B52",
    "spatial_static": "#3E7CB1",
    "spatial_temporal": "#1E9C89",
}
FAMILY_LABEL = {
    "local_static": "Local static",
    "local_temporal": "Local temporal",
    "spatial_static": "Spatial static",
    "spatial_temporal": "Spatial-temporal",
}
COMPONENT_COLOR = {
    "Grid power": "#7E8FA6",
    "Direct clean": "#2FAE60",
    "Storage": "#6449C6",
    "Routing": "#51F851",
    "Water": "#64B6C4",
}

def style(ax, grid_axis="y"):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    if grid_axis:
        ax.grid(axis=grid_axis, color="#E7ECF2", linewidth=0.8)
        ax.set_axisbelow(True)

def draw_radial_inset_split(ax, omega, table):
    order = ["local_static", "local_temporal", "spatial_static", "spatial_temporal"]
    one = table[(table["record_type"].eq("ring")) & np.isclose(table["omega"], omega)].copy()
    feasible = []
    for fam in order:
        rec = one[one["family"].eq(fam)]
        total = rec[rec["component"].eq("total")]["value_billion_cny"]
        if not total.empty and np.isfinite(total.iloc[0]):
            feasible.append((fam, rec, float(total.iloc[0])))
    if not feasible:
        return
    local_scale = max(total for _, _, total in feasible) * 1.03
    theta_start = np.deg2rad(100)
    theta_span = np.deg2rad(310)
    ring_h = 0.12
    base_r = 0.44
    gap = 0.14
    for idx, (fam, rec, _) in enumerate(feasible):
        r = base_r + (len(feasible) - 1 - idx) * gap
        cursor = theta_start
        values = rec.set_index("component")["value_billion_cny"].to_dict()
        for comp in ["Grid power", "Direct clean", "Storage", "Routing", "Water"]:
            val = float(values.get(comp, 0.0))
            width = theta_span * val / local_scale
            ax.bar(cursor + width / 2, ring_h, width=width, bottom=r,
                   color=COMPONENT_COLOR[comp], edgecolor="white", linewidth=0.25)
            cursor += width

    ax.text(0, 0.22, f"{int(omega*100)}%", ha="center", va="center", fontsize=7.2, weight="bold", color="#1F2937")
    ax.set_theta_direction(-1)
    ax.set_theta_offset(np.pi / 2)
    ax.set_ylim(0.34, 1.12)
    ax.set_yticks([])
    ax.set_xticks([])
    ax.set_xticklabels([])
    ax.grid(color="#E5E7EB", linewidth=0.35)
    ax.spines["polar"].set_color("#D1D5DB")

table = pd.read_feather(DATA)
table = table[table["atlas_table"].eq("main_figure3_a_split_power_rings_source")].dropna(axis=1, how="all").copy()
if table.empty:
    raise ValueError("main_figure3_a_split_power_rings_source is missing from nature_story_atlas.feather.")
path = table[table["record_type"].eq("path")].copy()

fig = plt.figure(figsize=(6.2, 4.6), dpi=300)
ax_container = fig.add_subplot(111)
ax_container.axis("off")

ax = ax_container.inset_axes([0.10, 0.45, 0.86, 0.48])

for fam in ["local_static", "local_temporal", "spatial_static", "spatial_temporal"]:
    sub = path[path["family"].eq(fam)].sort_values("omega")
    ax.plot(
        sub["omega"],
        sub["value_billion_cny"],
        marker="o",
        lw=2 if fam == "spatial_temporal" else 1.55,
        color=FAMILY_COLOR[fam],
        label=FAMILY_LABEL[fam],
    )

path_ymax = float(path["value_billion_cny"].max())

ax.axvspan(0.50, 0.82, color="#F4E7E1", alpha=0.65, zorder=0)
ax.text(0.515, path_ymax * 0.94, "Local cases\ninfeasible", color="#9B4A32", fontsize=7.5, weight="bold", va="top")

ax.set_ylabel("Total operating cost (billion CNY)")
ax.set_xlim(-0.02, 0.82)
style(ax)

for idx, omega in enumerate([0.0, 0.2, 0.4, 0.6, 0.8]):
    left = 0.03 + idx * 0.19
    ax_in = ax_container.inset_axes([left, 0.07, 0.17, 0.30], projection="polar")
    draw_radial_inset_split(ax_in, omega, table)

handles = [plt.Rectangle((0, 0), 1, 1, color=COMPONENT_COLOR[c])
           for c in ["Grid power", "Direct clean", "Storage", "Routing", "Water"]]
ax_container.legend(handles, ["Grid power", "Direct clean", "Storage service", "Compute routing", "Water cost"],
                    frameon=False, loc="lower center", bbox_to_anchor=(0.55, -0.02), ncol=3, fontsize=6.0)

fig.savefig(OUT, dpi=300, bbox_inches="tight")
plt.show()
pass


## Panel C

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / "data/nature_story_atlas.feather").exists():
    ROOT = Path.cwd() / "04_plotting_workspace" / "nature_story_atlas_complete_clean_20260702"
DATA = ROOT / "data/nature_story_atlas.feather"
OUT_DIR = ROOT / "figures" / "panels"
OUT_DIR.mkdir(parents=True, exist_ok=True)

mpl.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans", "sans-serif"],
    "font.size": 8,
    "axes.titlesize": 9,
    "axes.labelsize": 8,
    "xtick.labelsize": 7,
    "ytick.labelsize": 7,
    "legend.fontsize": 7,
    "axes.linewidth": 0.8,
    "svg.fonttype": "none",
    "pdf.fonttype": 42,
})

START_COLOR = "#334155"
END_COLOR = "#1E9C89"
UP_COLOR = "#C45A52"
DOWN_COLOR = "#5FA56F"
GRID = "#E7ECF2"
ORDER = ["pue", "grid_power", "green_power", "storage", "routing", "water"]
TICK_LABELS = ["from", "PUE", "grid", "cln.", "stor.", "rt.", "wat.", "to"]

def style(ax, grid_axis="y"):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    if grid_axis:
        ax.grid(axis=grid_axis, color=GRID, linewidth=0.8)
        ax.set_axisbelow(True)

def draw_waterfall(ax, sub, title):
    vals = sub.set_index("component")["value_billion_cny"].to_dict()
    start = vals["start"]
    end = vals["end"]
    x = np.arange(len(ORDER) + 2)
    bar_w = 0.88
    cursor = start

    ax.bar(x[0], start, color=START_COLOR, width=bar_w)
    ax.text(x[0], start + 1.5, f"{start:.1f}", ha="center", va="bottom", fontsize=5.2)

    for i, comp in enumerate(ORDER, start=1):
        delta = vals.get(comp, 0.0)
        y0 = cursor if delta >= 0 else cursor + delta
        ax.bar(x[i], abs(delta), bottom=y0, color=(UP_COLOR if delta > 0 else DOWN_COLOR), width=bar_w)

        if abs(delta) >= 0.25:

            ax.text(x[i], y0 + abs(delta) + 0.5, f"{delta:+.1f}", ha="center", va="bottom",
                    fontsize=4.5, color="#334155")
        cursor += delta

    ax.bar(x[-1], end, color=END_COLOR, width=bar_w)
    ax.text(x[-1], end + 1.5, f"{end:.1f}", ha="center", va="bottom", fontsize=5.2)

    ax.set_xticks(x)
    ax.set_xticklabels(TICK_LABELS, rotation=0, ha="center", fontsize=4.6)
    ax.tick_params(axis="y", labelsize=5.4, length=2)
    ax.set_xlim(-0.5, len(x) - 0.5)
    style(ax)

    ax.set_ylim(20, 82.2)

df = pd.read_feather(DATA)
df = df[df["atlas_table"].eq("supp_cost_flow_decomposition_source")].dropna(axis=1, how="all").copy()
df = df[df["dataset"].eq("mechanism_waterfall")].copy()

comparisons = [
    ("coordination_only_45_to_45_mechanism", "Coordination only\n45% -> 45%", "coordination_only"),
    ("coordination_plus_target_45_to_80_mechanism", "Coordination + target\n45% -> 80%", "coordination_plus_target"),
]

for name, title, file_suffix in comparisons:
    fig, ax = plt.subplots(figsize=(1.5, 2.2), dpi=300)

    draw_waterfall(ax, df[df["comparison"].eq(name)].copy(), title)

    fig.tight_layout(rect=[0, 0, 1, 0.96])

    out_png = OUT_DIR / f"supp_pue_split_waterfall_{file_suffix}.png"
    out_svg = OUT_DIR / f"supp_pue_split_waterfall_{file_suffix}.svg"
    fig.savefig(out_png, bbox_inches="tight")
    fig.savefig(out_svg, bbox_inches="tight")

    plt.show()
    pass


## Panel D

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt

ROOT = Path.cwd()
DATA = ROOT / "data/nature_story_atlas.feather"
if not DATA.exists():
    ROOT = Path.cwd() / "04_plotting_workspace" / "nature_story_atlas_complete_clean_20260702"
    DATA = ROOT / "data/nature_story_atlas.feather"
OUT = ROOT / "figures" / "panels" / "main_figure3_d_sensitivity_pue_only.png"
OUT.parent.mkdir(parents=True, exist_ok=True)

df_all = pd.read_feather(DATA)
raw = df_all[df_all["atlas_table"].eq("f4_sensitivities")].dropna(axis=1, how="all").copy()

mpl.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial', 'Helvetica', 'DejaVu Sans', 'sans-serif'],
    'font.size': 8,
    'axes.titlesize': 9,
    'axes.labelsize': 8,
    'xtick.labelsize': 7,
    'ytick.labelsize': 7,
    'legend.fontsize': 7,
    'axes.linewidth': 0.8,
    'svg.fonttype': 'none',
    'pdf.fonttype': 42,
})

def style(ax, grid_axis='y'):
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    if grid_axis:
        ax.grid(axis=grid_axis, color='#E7ECF2', linewidth=0.8)
        ax.set_axisbelow(True)

def add_rows(rows, label, mask, value_col='delta_cost_billion_cny'):
    sub = raw[mask].copy()
    for _, r in sub.iterrows():
        rows.append({'comparison': label, 'value': float(r[value_col])})

baseline = raw[raw['family'].eq('flex_sens_20')].iloc[0]
rows = []
add_rows(rows, 'Clean-power access', raw['family'].astype(str).str.startswith('greenaccess_sens_'))
add_rows(rows, 'Clean-power price', raw['family'].astype(str).str.startswith('greenprice_sens_'))
add_rows(rows, 'Hosting cap', raw['family'].astype(str).str.startswith('sharecap_sens_'))
add_rows(rows, 'Route cost', raw['family'].astype(str).str.startswith('routecost_sens_'))
add_rows(rows, 'Storage price', raw['family'].astype(str).str.startswith('storage_sens_'))
add_rows(rows, 'Flex share', raw['family'].astype(str).str.startswith('flex_sens_'))

puewe = raw[raw['family'].astype(str).str.startswith('puewe_sens_')].copy()
for _, r in puewe.iterrows():
    rows.append({'comparison': 'PUE', 'value': float(r['delta_cost_billion_cny'])})

sens = pd.DataFrame(rows)

fig, ax = plt.subplots(figsize=(1.03, 1.4), dpi=300)

order = [
    'Clean-power access',
    'Clean-power price',
    'Hosting cap',
    'Route cost',
    'Storage price',
    'Flex share',
    'PUE',
]

y_gap = 0.40
y_pos = np.arange(len(order)) * y_gap

global_xmin = sens['value'].min()
global_xmax = sens['value'].max()
x_range = global_xmax - global_xmin
x_pad = max(0.12 * x_range, 0.9)
x_left = global_xmin - x_pad
x_right = global_xmax + x_pad
ax.set_xlim(x_left, x_right)
label_x = x_left + 0.04 * (x_right - x_left)

for yi, name in zip(y_pos, order):
    sub = sens[sens['comparison'] == name]
    if sub.empty:
        continue
    xs = sub['value'].to_numpy(float)
    xmin = np.nanmin(xs)
    xmax = np.nanmax(xs)
    ax.plot([xmin, xmax], [yi, yi], color='#CBD5E1', lw=3.8, solid_capstyle='round', zorder=1, clip_on=False)
    for _, r in sub.iterrows():
        color = '#B3414A' if r['value'] > 0 else '#2F8F70'
        ax.scatter(r['value'], yi, s=38, color=color, edgecolor='white', linewidth=0.45, zorder=3, clip_on=False)
    ax.text(label_x, yi + 0.08, name, fontsize=5.0, fontweight='normal', color='#334155', va='bottom', ha='left')

ax.axvline(0, color='#1F2937', lw=0.6, zorder=0)
ax.set_yticks(y_pos)
ax.set_yticklabels([])
ax.tick_params(axis='y', length=0)
ax.set_xlabel('Cost change vs baseline\n(billion CNY)', fontsize=5.2)
ax.tick_params(axis='x', labelsize=5.0, length=2)
ax.set_title('d', loc='left', fontweight='bold', fontsize=6.0)
ax.set_ylim(y_pos[0] - 0.28, y_pos[-1] + 0.28)
style(ax, grid_axis='x')
fig.tight_layout(pad=0.35)
fig.savefig(OUT, bbox_inches='tight')
plt.show()
pass


## Panel D

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt

ROOT = Path.cwd()
DATA = ROOT / "data/nature_story_atlas.feather"
if not DATA.exists():
    ROOT = Path.cwd() / "04_plotting_workspace" / "nature_story_atlas_complete_clean_20260702"
    DATA = ROOT / "data/nature_story_atlas.feather"
OUT = ROOT / "figures" / "panels" / "main_figure3_d_sensitivity_pue_only_pct.png"
OUT.parent.mkdir(parents=True, exist_ok=True)

df_all = pd.read_feather(DATA)
raw = df_all[df_all["atlas_table"].eq("f4_sensitivities")].dropna(axis=1, how="all").copy()

mpl.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial', 'Helvetica', 'DejaVu Sans', 'sans-serif'],
    'font.size': 8,
    'axes.titlesize': 9,
    'axes.labelsize': 8,
    'xtick.labelsize': 7,
    'ytick.labelsize': 7,
    'legend.fontsize': 7,
    'axes.linewidth': 0.8,
    'svg.fonttype': 'none',
    'pdf.fonttype': 42,
})

def style(ax, grid_axis='y'):
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    if grid_axis:
        ax.grid(axis=grid_axis, color='#E7ECF2', linewidth=0.8)
        ax.set_axisbelow(True)

def pct_change(r):
    delta = float(r['delta_cost_billion_cny'])
    base_cost = float(r['objective_cny']) / 1e9 - delta
    return 100.0 * delta / base_cost

def add_rows(rows, label, mask):
    for _, r in raw[mask].iterrows():
        rows.append({'comparison': label, 'value': pct_change(r)})

baseline = raw[raw['family'].eq('flex_sens_20')].iloc[0]
assert np.isclose(float(baseline['delta_cost_billion_cny']), 0.0), "Baseline delta is not zero"

rows = []
add_rows(rows, 'Green access', raw['family'].astype(str).str.startswith('greenaccess_sens_'))
add_rows(rows, 'Green price', raw['family'].astype(str).str.startswith('greenprice_sens_'))
add_rows(rows, 'Hosting cap', raw['family'].astype(str).str.startswith('sharecap_sens_'))
add_rows(rows, 'Route cost', raw['family'].astype(str).str.startswith('routecost_sens_'))
add_rows(rows, 'Storage price', raw['family'].astype(str).str.startswith('storage_sens_'))
add_rows(rows, 'Flex share', raw['family'].astype(str).str.startswith('flex_sens_'))

add_rows(rows, 'PUE', raw['family'].astype(str).str.startswith('puewe_sens_'))

sens = pd.DataFrame(rows)

fig, ax = plt.subplots(figsize=(1.03, 1.4), dpi=300)

order = [
    'Green access',
    'Green price',
    'Hosting cap',
    'Route cost',
    'Storage price',
    'Flex share',
    'PUE',
]

y_gap = 0.40
y_pos = np.arange(len(order)) * y_gap

global_xmin = sens['value'].min()
global_xmax = sens['value'].max()
x_range = global_xmax - global_xmin
x_pad = max(0.12 * x_range, 0.3)
x_left = global_xmin - x_pad
x_right = global_xmax + x_pad
ax.set_xlim(x_left, x_right)
label_x = x_left + 0.04 * (x_right - x_left)

for yi, name in zip(y_pos, order):
    sub = sens[sens['comparison'] == name]
    if sub.empty:
        continue
    xs = sub['value'].to_numpy(float)
    xmin = np.nanmin(xs)
    xmax = np.nanmax(xs)
    ax.plot([xmin, xmax], [yi, yi], color='#CBD5E1', lw=3.8, solid_capstyle='round', zorder=1, clip_on=False)
    for _, r in sub.iterrows():
        color = '#B3414A' if r['value'] > 0 else '#2F8F70'
        ax.scatter(r['value'], yi, s=38, color=color, edgecolor='white', linewidth=0.45, zorder=3, clip_on=False)
    ax.text(label_x, yi + 0.08, name, fontsize=5.0, fontweight='normal', color='#334155', va='bottom', ha='left')

ax.axvline(0, color='#1F2937', lw=0.6, zorder=0)
ax.set_yticks(y_pos)
ax.set_yticklabels([])
ax.tick_params(axis='y', length=0)
ax.set_xlabel('Cost change vs baseline\n(%)', fontsize=5.2)
ax.tick_params(axis='x', labelsize=5.0, length=2)
ax.set_title('d', loc='left', fontweight='bold', fontsize=6.0)
ax.set_ylim(y_pos[0] - 0.28, y_pos[-1] + 0.28)
style(ax, grid_axis='x')
ax.xaxis.set_major_locator(mpl.ticker.MultipleLocator(4))
fig.tight_layout(pad=0.35)
fig.savefig(OUT, bbox_inches='tight')
plt.show()
pass

pass


## Panel E

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

CWD = Path.cwd()
DATA = CWD / 'figure25_coordination_effect_ONE_CSV.csv'
df = load_integrated_table('figure25_coordination_effect_ONE_CSV.csv')

def part(name):
    return df[df['dataset'].eq(name)].copy()

colors = {
    'local_static': '#C95E46',
    'local_temporal': '#F0A44B',
    'spatial_static': '#2E6E9E',
    'spatial_temporal': '#1B9A8A',
    'Grid': '#B8C1CC',
    'Direct clean': '#2ECC71',
    'Storage': '#E74C3C',
    'Routing': '#8E44AD',
}
plt.rcParams.update({'font.family':'DejaVu Sans','font.size':8,'axes.spines.top':False,'axes.spines.right':False})
pass

import os
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, Patch, Wedge

micro = part('figure25_coordination_effect_province_carbon_microbar_source').copy()
service = part('figure25_coordination_effect_province_service_supply_source').copy()

stage_order = [
    'local_static__S0_w0.45',
    'local_temporal__S0_w0.45',
    'spatial_static__S1_w0.80',
    'spatial_temporal__S1_w0.80',
]

stage_colors = ['#94A3B8', '#64748B', '#475569', '#1E293B']
service_ring_colors = {'local_exec': '#4A5568', 'imported_exec': '#D97706'}
supply_ring_colors = {'wind': '#059669', 'solar': '#F59E0B', 'hydro': '#06B6D4', 'grid': '#94A3B8'}

province_grid = {
    'Heilongjiang': (0, 10), 'Jilin': (1, 10), 'Liaoning': (2, 10),
    'Inner Mongolia': (1, 8), 'Beijing': (2, 7), 'Tianjin': (2, 8), 'Hebei': (3, 7), 'Shanxi': (3, 5),
    'Xinjiang': (3, 1), 'Gansu': (3, 3), 'Ningxia': (4, 4), 'Shaanxi': (4, 5), 'Qinghai': (4, 2),
    'Shandong': (3, 8), 'Jiangsu': (4, 9), 'Anhui': (5, 7), 'Shanghai': (4, 10), 'Zhejiang': (5, 9), 'Jiangxi': (6, 7), 'Fujian': (6, 9), 'Taiwan': (5, 10),
    'Henan': (4, 7), 'Hubei': (5, 5), 'Hunan': (6, 5),
    'Tibet': (5, 1), 'Sichuan': (5, 3), 'Chongqing': (5, 4), 'Guizhou': (6, 3), 'Yunnan': (6, 2),
    'Guangxi': (7, 4), 'Guangdong': (7, 5), 'Hainan': (8, 5), 'Hong Kong': (7, 8), 'Macao': (7, 7)
}

wide = micro.pivot_table(
    index=['province', 'province_en'], columns='case_key', values='carbon_mtco2', aggfunc='first'
).reset_index()
wide = wide.merge(service.drop(columns=[c for c in ['province_en', 'lon', 'lat'] if c in service.columns]), on='province', how='left')

all_values = wide[stage_order].values.flatten()
pos_values = all_values[all_values > 0]
global_ymin = float(np.min(pos_values)) * 0.5 if len(pos_values) > 0 else 1e-3
global_ymax = float(np.max(all_values)) * 1.25
max_exec = max(float(wide['executed_total_it_twh'].max()), 1e-9)

output_dir = "exported_tiles"
os.makedirs(output_dir, exist_ok=True)

pass

for _, row in wide.iterrows():
    p_en = row['province_en']
    if p_en not in province_grid:
        continue

    bg_color = '#F8FAFC90'

    if p_en in ['Guangxi', 'Guangdong', 'Hainan', 'Hong Kong', 'Macao']:
        bg_color = "#b6dbb494"

    elif p_en in ['Yunnan', 'Guizhou', 'Sichuan', 'Chongqing', 'Tibet']:
        bg_color = "#e2ff9192"

    elif p_en in ['Qinghai', 'Ningxia', 'Gansu', 'Xinjiang', 'Shaanxi']:
        bg_color = "#d8bfa592"

    elif p_en in ['Beijing', 'Shanxi', 'Hebei', 'Tianjin', 'Inner Mongolia']:
        bg_color = "#ffdbdc92"

    elif p_en in ['Heilongjiang', 'Jilin', 'Liaoning']:
        bg_color = "#d4d3d692"

    elif p_en in ['Shandong', 'Jiangsu', 'Shanghai','Zhejiang','Anhui', 'Jiangxi', 'Fujian', 'Taiwan']:
        bg_color = "#cde0f894"

    elif p_en in ['Hunan', 'Hubei', 'Henan']:
        bg_color = "#faefcd94"

    fig, ax = plt.subplots(figsize=(1.5, 1.1), dpi=300)
    ax.set_xlim(0, 1)
    ax.set_ylim(0.78, 0)
    ax.set_aspect('equal')
    ax.axis('off')

    rect = Rectangle(
        (0.04, 0.04), 0.92, 0.70,
        facecolor=bg_color, edgecolor='#F1F5F9FF', linewidth=0.6, zorder=1
    )
    ax.add_patch(rect)

    ax_bar = ax.inset_axes([0.09, 0.08, 0.82, 0.52])

    vals = np.array([float(row[k]) for k in stage_order], dtype=float)

    ax_bar.bar(range(len(vals)), np.maximum(vals - global_ymin, 0.0), bottom=global_ymin, color=stage_colors, width=0.72, edgecolor='none', zorder=2)
    ax_bar.set_yscale('log')
    ax_bar.set_ylim(global_ymin, global_ymax)
    ax_bar.axis('off')
    ax_bar.axhline(global_ymin, color='#CBD5E1', linewidth=0.4, zorder=1)

    for zero_i in np.flatnonzero(vals == 0):
        ax_bar.hlines(global_ymin, zero_i - 0.36, zero_i + 0.36,
                      colors=stage_colors[zero_i], linewidth=1.0, zorder=3, clip_on=False)
    local_mean = float(np.mean(vals[:2]))
    spatial_mean = float(np.mean(vals[2:]))
    arrow_color = '#EF4444' if spatial_mean > local_mean else '#10B981'

    vals_pos = np.maximum(vals, global_ymin)
    y_arrow = float(np.exp((np.log(np.mean(vals_pos[:2])) + np.log(np.mean(vals_pos[2:]))) / 2.0))
    rad = 0.22 if spatial_mean > local_mean else -0.22

    ax_bar.annotate(
        '', xy=(2.55, y_arrow), xytext=(0.45, y_arrow),
        arrowprops=dict(
            arrowstyle='->', color=arrow_color, lw=1.5, alpha=1.0,
            connectionstyle=f"arc3,rad={rad}", shrinkA=0.6, shrinkB=0.6, mutation_scale=5.5
        ),
        zorder=4,
    )
    ax_bar.set_facecolor('none')

    scale = np.sqrt(max(float(row['executed_total_it_twh']), 0.0) / max_exec)
    r_max = 0.16 + 0.19 * scale

    center_x, center_y = 0.04, 0.04

    inner_raw = np.array([float(row['local_exec_share']), float(row['imported_exec_share'])], dtype=float)
    outer_raw = np.array([float(row['wind_supply_share']), float(row['solar_supply_share']), float(row['hydro_supply_share']), float(row['grid_supply_share'])], dtype=float)

    if inner_raw.sum() > 0: inner_raw /= inner_raw.sum()
    if outer_raw.sum() > 0: outer_raw /= outer_raw.sum()

    if outer_raw.sum() > 0:
        current_angle = 0.0
        outer_colors_list = [supply_ring_colors['wind'], supply_ring_colors['solar'], supply_ring_colors['hydro'], supply_ring_colors['grid']]
        for val, color in zip(outer_raw, outer_colors_list):
            if val <= 0: continue
            delta_angle = val * 90.0
            wedge = Wedge(
                center=(center_x, center_y), r=r_max, theta1=current_angle, theta2=current_angle + delta_angle,
                width=r_max * 0.36, facecolor=color, edgecolor='white', linewidth=0.25, zorder=3, alpha=1.0
            )
            ax.add_patch(wedge)
            current_angle += delta_angle

    if inner_raw.sum() > 0:
        current_angle = 0.0
        inner_colors_list = [service_ring_colors['local_exec'], service_ring_colors['imported_exec']]
        r_inner = r_max * 0.64
        for val, color in zip(inner_raw, inner_colors_list):
            if val <= 0: continue
            delta_angle = val * 90.0
            wedge = Wedge(
                center=(center_x, center_y), r=r_inner, theta1=current_angle, theta2=current_angle + delta_angle,
                width=r_inner * 0.45, facecolor=color, edgecolor='white', linewidth=0.25, zorder=3, alpha=1.0
            )
            ax.add_patch(wedge)
            current_angle += delta_angle

    ax.text(
        0.5, 0.24, p_en,
        fontsize=6.5, fontweight='bold', color='#475569',
        ha='center', va='center', zorder=5, alpha=1.0
    )

    safe_name = p_en.replace(" ", "_")
    file_path = os.path.join(output_dir, f"tile_{safe_name}.png")

    plt.savefig(file_path, bbox_inches='tight', pad_inches=0.01, transparent=True, dpi=300)
    plt.close(fig)

pass


## Panel F

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

CWD = Path.cwd()
DATA = CWD / 'figure25_coordination_effect_ONE_CSV.csv'
df = load_integrated_table('figure25_coordination_effect_ONE_CSV.csv')

def part(name):
    return df[df['dataset'].eq(name)].copy()

colors = {
    'local_static': '#C95E46',
    'local_temporal': '#F0A44B',
    'spatial_static': '#2E6E9E',
    'spatial_temporal': '#1B9A8A',
    'Grid': '#B8C1CC',
    'Direct clean': '#2ECC71',
    'Storage': '#E74C3C',
    'Routing': '#8E44AD',
}
plt.rcParams.update({'font.family':'DejaVu Sans','font.size':8,'axes.spines.top':False,'axes.spines.right':False})
pass
from matplotlib.patches import Patch
import numpy as np
import matplotlib.pyplot as plt

sankey = part('figure25_coordination_effect_sankey_source').copy()
stage = part('figure25_coordination_effect_stage_metrics_source').copy().sort_values('stage')
component_order = ['Grid', 'Direct clean', 'Storage', 'Routing']
component_colors = {
    'Grid': '#AEB6BF',
    'Direct clean': '#2ECC71',
    'Storage': '#E64B35',
    'Routing': '#8E44AD',
}

def draw_stacked_band(ax, x1, x2, y1_start, y1_end, y2_start, y2_end, color, alpha=0.76):
    x = np.linspace(x1, x2, 120)
    t = (x - x1) / (x2 - x1)
    lower = ((1 - t) ** 3 * y1_start + 3 * (1 - t) ** 2 * t * y1_start + 3 * (1 - t) * t**2 * y2_start + t**3 * y2_start)
    upper = ((1 - t) ** 3 * y1_end + 3 * (1 - t) ** 2 * t * y1_end + 3 * (1 - t) * t**2 * y2_end + t**3 * y2_end)
    ax.fill_between(x, lower, upper, color=color, alpha=alpha, edgecolor='none')

share_lookup = {(int(r.stage), r.component): float(r.share) for r in sankey.itertuples(index=False)}
water_values = stage['total_water_m3'].to_numpy(dtype=float) / 1e6
water_min = float(np.nanmin(water_values))
water_span = max(float(np.nanmax(water_values) - water_min), 1e-9)
x_anchors = np.array([0.2, 1.2, 2.2, 3.2], dtype=float)
y0, height = 0.1, 1.0
bounds = {}
for i in range(len(stage)):
    current = y0
    for comp in component_order:
        share = share_lookup[(i, comp)]
        bounds[(i, comp)] = (current, current + share * height)
        current += share * height

fig, ax = plt.subplots(figsize=(5.0, 3.6), dpi=180)
ax.set_xlim(-0.2, 4.3)
ax.set_ylim(-0.2, 1.2)
ax.axis('off')

for i in range(len(stage) - 1):
    x1 = x_anchors[i] + 0.02
    x2 = x_anchors[i + 1] - 0.02
    for comp in component_order:
        y1s, y1e = bounds[(i, comp)]
        y2s, y2e = bounds[(i + 1, comp)]
        draw_stacked_band(ax, x1, x2, y1s, y1e, y2s, y2e, component_colors[comp])

for i, (_, row) in enumerate(stage.iterrows()):
    x = x_anchors[i]
    ax.axvline(x, color='#2C3E50', linestyle=':', alpha=0.4, zorder=1)
    ax.fill_between([x - 0.02, x + 0.02], [0.1, 0.1], [1.1, 1.1], color='#2C3E50', alpha=0.15, zorder=2)

    ax.text(x, 1.13, row['stage_label'].replace('\n', ' '), ha='center', va='bottom', fontsize=8.2, fontweight='bold', color='#2C3E50')

    if i > 0:
        valid_labels = []
        for comp in component_order:
            ys, ye = bounds[(i, comp)]
            share = share_lookup[(i, comp)]
            if share >= 0.025:
                valid_labels.append({'comp': comp, 'center': (ys + ye) / 2, 'share': share})

        for idx, item in enumerate(valid_labels):
            comp_name = item['comp']
            share_val = item['share']
            y_pos = item['center']

            x_offset = -0.07
            ha_align = 'right'
            text_color = 'white'

            if share_val < 0.05:

                if idx % 2 == 0:
                    x_offset = -0.08
                    ha_align = 'right'
                else:
                    x_offset = 0.08
                    ha_align = 'left'
                text_color = component_colors[comp_name]

            ax.text(
                x + x_offset,
                y_pos,
                f'{share_val * 100:.0f}%',
                ha=ha_align,
                va='center',
                fontsize=8.0,
                fontweight='bold',
                color=text_color
            )

    h_water = 0.025 + (row['total_water_m3'] / 1e6 - water_min) / water_span * 0.075
    ax.fill_between([x - 0.04, x + 0.04], [-0.15, -0.15], [-0.15 + h_water, -0.15 + h_water], color='#2980B9', alpha=0.82, zorder=4)

    if i < len(stage) - 1:
        next_row = stage.iloc[i + 1]
        next_h = 0.025 + (next_row['total_water_m3'] / 1e6 - water_min) / water_span * 0.075
        draw_stacked_band(ax, x + 0.04, x_anchors[i + 1] - 0.04, -0.15, -0.15 + h_water, -0.15, -0.15 + next_h, color='#2980B9', alpha=0.42)

    ax.text(x, -0.19, f"{row['total_water_m3'] / 1e6:.1f}", ha='center', va='top', fontsize=6.8, color='#2980B9', fontweight='bold')

handles = [Patch(facecolor=component_colors[c], label=c) for c in component_order]
handles.append(Patch(facecolor='#2980B9', label='Water footprint'))

ax.legend(handles=handles, frameon=False, ncol=1, loc='center right', bbox_to_anchor=(1.28, 0.50), fontsize=7)

ax.text(-0.08, 0.6, 'Normalized\ncoordination\nstructure', color='#2C3E50', fontsize=7.5, fontweight='bold', ha='right', va='center')
ax.text(-0.08, -0.12, 'Water\nfootprint\n(Mm3)', color='#2980B9', fontsize=7.3, fontweight='bold', ha='right', va='center')

plt.show()


## Panel G

In [ ]:
import struct
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection
from matplotlib.patches import FancyArrowPatch
import matplotlib.patheffects as path_effects
from pathlib import Path
import json as _integrated_json
import pandas as pd
from pyproj import CRS, Transformer

INTEGRATED_DATA = Path.cwd() / "data/current_results_legacy_plot_schema.feather"
_bundle = pd.read_feather(INTEGRATED_DATA)
_rows = _bundle[_bundle["source_csv"].eq("figure28_regional_carbon_transfer_ONE_CSV.csv")].copy()
if _rows.empty:
    raise FileNotFoundError("Missing integrated table: figure28_regional_carbon_transfer_ONE_CSV.csv")
_prefix = _rows["source_prefix"].iloc[0]
_dtype_map = _integrated_json.loads(_rows["source_dtypes_json"].iloc[0])
_columns = [c for c in _rows.columns if c.startswith(_prefix)]
_regional = _rows[_columns].copy()
_regional.columns = [c[len(_prefix):] for c in _columns]
for _column, _dtype in _dtype_map.items():
    if _dtype.startswith("int"):
        _regional[_column] = pd.to_numeric(_regional[_column], errors="raise").astype(_dtype)
    elif _dtype.startswith("float"):
        _regional[_column] = pd.to_numeric(_regional[_column], errors="coerce").astype(_dtype)
    elif _dtype == "bool":
        _regional[_column] = _regional[_column].astype(bool)
    elif _dtype == "object":
        _regional[_column] = _regional[_column].where(_regional[_column].notna(), None).astype(object)
node_df = _regional[_regional["dataset"].eq("figure28_regional_carbon_transfer_nodes")].copy()
edge_df = _regional[_regional["dataset"].eq("figure28_regional_carbon_transfer_edges")].copy()
if node_df.empty or edge_df.empty:
    raise ValueError("Regional carbon-transfer node or edge rows are missing")

_china_gk = CRS.from_proj4("+proj=tmerc +lat_0=0 +lon_0=105 +k=1 +x_0=500000 +y_0=0 +ellps=GRS80 +units=m +no_defs")
_transformer = Transformer.from_crs("EPSG:4490", _china_gk, always_xy=True)
def project_china_lonlat(lon, lat):
    x_m, y_m = _transformer.transform(np.asarray(lon, dtype=float), np.asarray(lat, dtype=float))
    return np.asarray(x_m) / 1000.0, np.asarray(y_m) / 1000.0

shp = Path.cwd() / 'boundaries' / "bou2_4l.shp"
if not shp.is_file():
    raise FileNotFoundError('Missing local boundary: boundaries/bou2_4l.shp')

segments = []

OWN_COLOR = '#5C7162'
EXTERNAL_COLOR = '#BD7F6A'
EDGE_BASE_COLOR = "#75A4DD"
EDGE_STROKE_COLOR = "#FEFEFF"

if shp is not None and shp.exists():
    with shp.open('rb') as fh:
        fh.seek(100)
        while True:
            header = fh.read(8)
            if len(header) < 8:
                break
            _, content_words = struct.unpack('>2i', header)
            content = fh.read(content_words * 2)
            if len(content) < 44:
                continue
            shape_type = struct.unpack('<i', content[:4])[0]
            if shape_type not in (3, 5):
                continue
            num_parts, num_points = struct.unpack('<2i', content[36:44])
            if num_parts <= 0 or num_points <= 1:
                continue
            parts_offset = 44
            points_offset = parts_offset + 4 * num_parts
            parts = list(struct.unpack(f'<{num_parts}i', content[parts_offset:points_offset]))
            parts.append(num_points)
            raw_points = np.frombuffer(content[points_offset:points_offset + 16 * num_points], dtype='<f8').reshape(-1, 2)

            x, y = project_china_lonlat(raw_points[:, 0], raw_points[:, 1])
            points = np.column_stack([x, y])
            for start, end in zip(parts[:-1], parts[1:]):
                if end - start >= 2:
                    segments.append(points[start:end])

all_points = np.vstack(segments)
xmin, xmax = float(np.nanmin(all_points[:, 0])), float(np.nanmax(all_points[:, 0]))
ymin, ymax = float(np.nanmin(all_points[:, 1])), float(np.nanmax(all_points[:, 1]))
xpad = (xmax - xmin) * 0.035
ypad = (ymax - ymin) * 0.04

def draw_node_pie(ax, x, y, radius, own_val, external_val):
    pie_ax = ax.inset_axes([x - radius, y - radius, radius * 2, radius * 2], transform=ax.transData)
    vals = np.asarray([max(own_val, 0.0), max(external_val, 0.0)], dtype=float)
    if vals.sum() <= 0:
        vals = np.asarray([1.0, 0.0])

    pie_ax.pie(vals, startangle=90, counterclock=False, colors=[OWN_COLOR, EXTERNAL_COLOR],
               wedgeprops=dict(edgecolor='white', linewidth=1.2, alpha=1.0))
    pie_ax.set_aspect('equal')
    pie_ax.set_xticks([])
    pie_ax.set_yticks([])
    for spine in pie_ax.spines.values():
        spine.set_visible(False)
    pie_ax.set_facecolor((1, 1, 1, 0))

fig, ax = plt.subplots(figsize=(12.8, 8.4), dpi=1200)
ax.set_xlim(xmin - xpad, xmax + xpad)
ax.set_ylim(ymin - ypad, ymax + ypad)
ax.set_aspect('equal', adjustable='box', anchor='C')
ax.set_xticks([])
ax.set_yticks([])

for spine in ax.spines.values():
    spine.set_visible(False)

fig.patch.set_alpha(0.0)
ax.patch.set_alpha(0.0)

ax.add_collection(LineCollection(segments, colors='#CBD5E1', linewidths=0.5, alpha=0.0, zorder=0))

node_df['total_carbon_mtco2'] = node_df['total_carbon_mtco2'].astype(float)
total_max = max(float(node_df['total_carbon_mtco2'].max()) if not node_df.empty else 1.0, 1e-6)

node_radius_dict = {}
for _, row in node_df.iterrows():
    r = 180.0 + 170.0 * np.sqrt(float(row['total_carbon_mtco2']) / total_max)
    node_radius_dict[(round(float(row['x_km']), 2), round(float(row['y_km']), 2))] = r

def get_node_radius(x, y):
    key = (round(float(x), 2), round(float(y), 2))
    return node_radius_dict.get(key, 265.0)

edge_df['carbon_transfer_mtco2'] = edge_df['carbon_transfer_mtco2'].astype(float)
edge_max = max(float(edge_df['carbon_transfer_mtco2'].max()) if not edge_df.empty else 1.0, 1e-6)

sorted_edges = edge_df.sort_values('carbon_transfer_mtco2')

FIXED_STROKE_WIDTH = 1.0

for i, (_, row) in enumerate(sorted_edges.iterrows()):
    x1, y1 = float(row['from_x_km']), float(row['from_y_km'])
    x2, y2 = float(row['to_x_km']), float(row['to_y_km'])
    val = float(row['carbon_transfer_mtco2'])

    r_from = get_node_radius(x1, y1)
    r_to = get_node_radius(x2, y2)

    base_shrinkA = r_from * 0.075 - 0.2
    base_shrinkB = r_to * 0.075 - 0.2

    arrow_lw = 3.5 + 6.5 * (val / edge_max)
    base_z = 1 + (i * 0.01)

    style_str = f"simple,tail_width={arrow_lw:.2f},head_width={arrow_lw*2.1:.2f},head_length={arrow_lw*2.4:.2f}"

    arrow = FancyArrowPatch(
        (x1, y1), (x2, y2),
        arrowstyle=style_str,
        mutation_scale=1.0,
        connectionstyle="arc3,rad=0",
        facecolor=EDGE_BASE_COLOR,
        edgecolor='none',
        alpha=1.0,
        shrinkA=base_shrinkA,
        shrinkB=base_shrinkB,
        capstyle='butt',
        joinstyle='miter',
        zorder=base_z
    )

    arrow.set_path_effects([
        path_effects.Stroke(linewidth=1.0 + (FIXED_STROKE_WIDTH * 2.0), foreground=EDGE_STROKE_COLOR),
        path_effects.Normal()
    ])

    ax.add_patch(arrow)

for _, row in node_df.iterrows():
    x, y = float(row['x_km']), float(row['y_km'])
    radius = get_node_radius(x, y)

    draw_node_pie(ax, x, y, radius, float(row['own_demand_carbon_mtco2']), float(row['external_service_carbon_mtco2']))

output_path = Path.cwd() / "figures" / "panels" / "Figure3_g.png"

output_path.parent.mkdir(parents=True, exist_ok=True)

plt.savefig(output_path, transparent=True, dpi=1200, bbox_inches='tight')

pass
pass

plt.show()


## Panel H

In [ ]:
from pathlib import Path
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Image, display

ROOT = Path.cwd()
if not (ROOT / "data/nature_story_atlas.feather").exists():
    ROOT = Path.cwd() / "04_plotting_workspace" / "nature_story_atlas_complete_clean_20260702"
DATA = ROOT / "data/nature_story_atlas.feather"
OUT_DIR = ROOT / "figures" / "panels" / "environment_candidates"
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT = OUT_DIR / "env_cw_tradeoff_v2.png"
OUT_SVG = OUT_DIR / "env_cw_tradeoff_v2.svg"

df = pd.read_feather(DATA)
df = df[df["atlas_table"].eq("main_figure3_environment_carbon_water_stress_score_source")].copy()
if df.empty:
    raise ValueError("Missing atlas_table: main_figure3_environment_carbon_water_stress_score_source")

region_color = {
    "East": "#D7796A",
    "North China": "#9B7BB7",
    "Northeast": "#8EA0AD",
    "Central": "#E0A12B",
    "South": "#69A88C",
    "South China": "#69A88C",
    "Southwest": "#2F9E92",
    "Northwest": "#6D8FC7",
}
label_map = {
    "Beijing": "Beijing", "Shanghai": "Shanghai", "Guangdong": "Guangdong",
    "Jiangsu": "Jiangsu", "Zhejiang": "Zhejiang", "Inner Mongolia": "Inner M.",
    "Xinjiang": "Xinjiang", "Yunnan": "Yunnan", "Sichuan": "Sichuan",
}
label_offsets = {
    "Yunnan": (-0.64, 3.5), "Inner Mongolia": (0.22, 3.0), "Xinjiang": (0.20, 1.6),
    "Sichuan": (0.20, -2.3), "Beijing": (-0.55, -0.9), "Guangdong": (0.20, -0.8),
    "Shanghai": (0.18, 1.1), "Jiangsu": (0.15, 0.7), "Zhejiang": (0.15, 0.6),
}

mpl.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans", "sans-serif"],
    "font.size": 8,
    "axes.titlesize": 9,
    "axes.labelsize": 9,
    "xtick.labelsize": 8.5,
    "ytick.labelsize": 8.5,
    "legend.fontsize": 7,
    "axes.linewidth": 0.8,
    "svg.fonttype": "none",
    "pdf.fonttype": 42,
})

x = df["carbon_change_mtco2"].astype(float)
y = df["water_change_million_m3"].astype(float)
size = df["service_size_twh"].astype(float).clip(lower=0.2)
sizes = 16 + 180 * (size / size.max()) ** 0.65

fig, ax = plt.subplots(figsize=(4.5, 2.6), dpi=300)

x_min_val, x_max_val = x.min() - 0.9, x.max() + 0.9
y_min_val, y_max_val = y.min() - 6, y.max() + 12

ax.fill_between([0, x_max_val], 0, y_max_val, color="#FEF08A", alpha=0.5, zorder=0)

ax.fill_between([x_min_val, 0], 0, y_max_val, color="#FFFFFF", alpha=0.3, zorder=0)

ax.fill_between([x_min_val, 0], y_min_val, 0, color="#E0F2FE", alpha=0.5, zorder=0)

ax.fill_between([0, x_max_val], y_min_val, 0, color="#FFFFFF", alpha=0.5, zorder=0)

for region, sub in df.groupby("region_en", sort=False):
    ax.scatter(
        sub["carbon_change_mtco2"], sub["water_change_million_m3"],
        s=sizes[sub.index], color=region_color.get(region, "#9AA6B2"),
        edgecolor="white", linewidth=0.55, alpha=0.92, label=region, zorder=3,
    )

for _, row in df.iterrows():
    name = row["province_en"]
    if name in label_map:
        dx, dy = label_offsets.get(name, (0.18, 0.8))
        ax.text(row["carbon_change_mtco2"] + dx, row["water_change_million_m3"] + dy,
                label_map[name], fontsize=8, color="#2E3A46", zorder=4)

ax.set_xlim(x.min() - 0.9, x.max() + 0.9)
ax.set_ylim(y.min() - 6, y.max() + 12)

ax.set_xlabel("Change in operational carbon emissions (MtCO$_2$)")
ax.set_ylabel("Change in water footprint (million m$^3$)")
ax.grid(color="#E7ECF2", lw=0.7, zorder=0)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

handles, labels = ax.get_legend_handles_labels()
seen = {}
for h, lab in zip(handles, labels):
    if lab and lab not in seen:
        seen[lab] = h

for s0, lab, xpos in [(2, "2", 0.08), (8, "8", 0.17), (16, "16", 0.28)]:
    ax.scatter([xpos], [0.92], s=16 + 180 * (s0 / size.max()) ** 0.65, transform=ax.transAxes,
               color="#CBD5E1", edgecolor="white", linewidth=0.5, zorder=5)
    ax.text(xpos, 0.845, lab, transform=ax.transAxes, ha="center", va="top", fontsize=8, color="#667786")
ax.text(0.055, 0.995, "hosted service (TWh-IT)", transform=ax.transAxes, ha="left", va="top", fontsize=8, color="#667786")

fig.tight_layout(pad=0.6)
fig.savefig(OUT, bbox_inches="tight")
fig.savefig(OUT_SVG, bbox_inches="tight")
plt.show()

pass


## Panel I

In [ ]:
from pathlib import Path
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Image, display

ROOT = Path.cwd()
if not (ROOT / "data/nature_story_atlas.feather").exists():
    ROOT = Path.cwd() / "04_plotting_workspace" / "nature_story_atlas_complete_clean_20260702"
DATA = ROOT / "data/nature_story_atlas.feather"
OUT_DIR = ROOT / "figures" / "panels" / "environment_candidates"
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT = OUT_DIR / "env_water_pressure_rank_v3.png"
OUT_SVG = OUT_DIR / "env_water_pressure_rank_v3.svg"

df = pd.read_feather(DATA)
df = df[df["atlas_table"].eq("main_figure3_environment_carbon_water_stress_score_source")].copy()
if df.empty:
    raise ValueError("Missing atlas_table: main_figure3_environment_carbon_water_stress_score_source")

region_bg = {
    "East": "#F6D7CF",
    "North China": "#E4D8F0",
    "Northeast": "#DDE5EC",
    "Central": "#F7E7BF",
    "South": "#D9EDE4",
    "South China": "#D9EDE4",
    "Southwest": "#D8EEE9",
    "Northwest": "#D8E4F5",
}

df["weighted_max_abs_score"] = df[["natural_weighted_water_score", "socio_weighted_water_score"]].abs().max(axis=1)

df = df.sort_values("weighted_max_abs_score", ascending=False).reset_index(drop=True)
y = np.arange(len(df))

mpl.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans", "sans-serif"],
    "font.size": 8,
    "axes.titlesize": 8,
    "axes.labelsize": 7,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "legend.fontsize": 6,
    "axes.linewidth": 0.8,
    "svg.fonttype": "none",
    "pdf.fonttype": 42,
})

fig, ax = plt.subplots(figsize=(2.9, 5), dpi=180)

for yi, region in zip(y, df["region_en"]):
    current_color = region_bg.get(region, "#F1F5F9")
    ax.axhspan(
        yi - 0.5, yi + 0.5,
        color=current_color,
        alpha=0.56,
        edgecolor=current_color,
        linewidth=0.5,
        zorder=0
    )

ax.axvline(0, color="#2F3B48", lw=0.9, zorder=1)

ax.hlines(
    y,
    df["natural_weighted_water_score"],
    df["socio_weighted_water_score"],
    colors="#94A3B8",
    linewidth=1.2,
    zorder=2
)

ax.scatter(df["natural_weighted_water_score"], y, s=32, color="#4C78A8", edgecolor="white", linewidth=0.6,
           label="Natural score", zorder=3)
ax.scatter(df["socio_weighted_water_score"], y, s=32, color="#C05A4B", edgecolor="white", linewidth=0.6,
           label="Socio-economic score", zorder=3)

ax.set_yticks(y)
ax.set_yticklabels(df["province_en"].replace({"Inner Mongolia": "Nei Mongol"}).tolist())

ax.set_ylim(len(df) - 0.5, -0.5)

ax.yaxis.tick_right()
ax.yaxis.set_label_position("right")
ax.tick_params(axis="y", pad=5, length=0)
ax.spines["left"].set_visible(False)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(True)
ax.spines["bottom"].set_visible(True)

ax.grid(axis="x", color="#E7ECF2", lw=0.7)
ax.set_axisbelow(True)

score_cols = ["natural_weighted_water_score", "socio_weighted_water_score"]
xmin = float(df[score_cols].min().min())
xmax = float(df[score_cols].max().max())
span = xmax - xmin
ax.set_xlim(xmin - 0.08 * span, xmax + 0.10 * span)
ax.set_xlabel("Score-weighted water change\n(million m$^3$ x stress score)")

ax.legend(loc="lower left", bbox_to_anchor=(0.02, 0.01), frameon=False, handletextpad=0.35)

fig.tight_layout(pad=0.55)
fig.savefig(OUT, bbox_inches="tight")
fig.savefig(OUT_SVG, bbox_inches="tight")
plt.show()

pass


## Panel I

In [ ]:
from pathlib import Path
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Image, display

ROOT = Path.cwd()
if not (ROOT / "data/nature_story_atlas.feather").exists():
    ROOT = Path.cwd() / "04_plotting_workspace" / "nature_story_atlas_complete_clean_20260702"
DATA = ROOT / "data/nature_story_atlas.feather"
OUT_DIR = ROOT / "figures" / "panels" / "environment_candidates"
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT = OUT_DIR / "env_water_pressure_rank_v3_top20.png"
OUT_SVG = OUT_DIR / "env_water_pressure_rank_v3_top20.svg"

TOP_N = 20

df = pd.read_feather(DATA)
df = df[df["atlas_table"].eq("main_figure3_environment_carbon_water_stress_score_source")].copy()
if df.empty:
    raise ValueError("Missing atlas_table: main_figure3_environment_carbon_water_stress_score_source")

region_bg = {
    "East": "#F6D7CF",
    "North China": "#E4D8F0",
    "Northeast": "#DDE5EC",
    "Central": "#F7E7BF",
    "South": "#D9EDE4",
    "South China": "#D9EDE4",
    "Southwest": "#D8EEE9",
    "Northwest": "#D8E4F5",
}

df["weighted_max_abs_score"] = df[["natural_weighted_water_score", "socio_weighted_water_score"]].abs().max(axis=1)

df = df.sort_values("weighted_max_abs_score", ascending=False).head(TOP_N).reset_index(drop=True)
y = np.arange(len(df))

mpl.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans", "sans-serif"],
    "font.size": 11,
    "axes.titlesize": 11,
    "axes.labelsize": 10,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
    "legend.fontsize": 9,
    "axes.linewidth": 0.8,
    "svg.fonttype": "none",
    "pdf.fonttype": 42,
})

fig, ax = plt.subplots(figsize=(2.9, 5), dpi=180)

for yi, region in zip(y, df["region_en"]):
    current_color = region_bg.get(region, "#F1F5F9")
    ax.axhspan(
        yi - 0.5, yi + 0.5,
        color=current_color,
        alpha=0.56,
        edgecolor=current_color,
        linewidth=0.5,
        zorder=0
    )

ax.axvline(0, color="#2F3B48", lw=0.9, zorder=1)

ax.hlines(
    y,
    df["natural_weighted_water_score"],
    df["socio_weighted_water_score"],
    colors="#94A3B8",
    linewidth=1.2,
    zorder=2
)

ax.scatter(df["natural_weighted_water_score"], y, s=32, color="#4C78A8", edgecolor="white", linewidth=0.6,
           label="Natural score", zorder=3)
ax.scatter(df["socio_weighted_water_score"], y, s=32, color="#C05A4B", edgecolor="white", linewidth=0.6,
           label="Socio-economic score", zorder=3)

ax.set_yticks(y)
ax.set_yticklabels(df["province_en"].replace({"Inner Mongolia": "Nei Mongol"}).tolist())
ax.set_ylim(len(df) - 0.5, -0.5)

ax.yaxis.tick_right()
ax.yaxis.set_label_position("right")
ax.tick_params(axis="y", pad=5, length=0)
ax.spines["left"].set_visible(False)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(True)
ax.spines["bottom"].set_visible(True)

ax.grid(axis="x", color="#E7ECF2", lw=0.7)
ax.set_axisbelow(True)

score_cols = ["natural_weighted_water_score", "socio_weighted_water_score"]
xmin = float(df[score_cols].min().min())
xmax = float(df[score_cols].max().max())
span = xmax - xmin
ax.set_xlim(xmin - 0.08 * span, xmax + 0.10 * span)
ax.set_xlabel("Score-weighted water change\n(million m$^3$ x stress score)")

fig.tight_layout(pad=0.55)
fig.savefig(OUT, bbox_inches="tight")
fig.savefig(OUT_SVG, bbox_inches="tight")
plt.show()

pass


## Panel J

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

CWD = Path.cwd()

def locate_file(name: str) -> Path:
    direct = CWD / name
    if not direct.exists():
        raise FileNotFoundError(f"Missing current-branch input: {name}")
    return direct

DATA = INTEGRATED_DATA
df = load_integrated_table('figure26_equity_exposure_ONE_CSV.csv')

def part(name):
    return df[df['dataset'].eq(name)].copy()

COLORS = {
    'Local static 45%': '#C95E46',
    'Local temporal 45%': '#F0A44B',
    'Spatial static 80%': '#2E6E9E',
    'Spatial-temporal 80%': '#1B9A8A',
    'East': '#4C78A8',
    'Central': '#F2B84B',
    'West': '#59A14F',
    'Northeast': '#9C755F',
}

def lorenz(values: np.ndarray, weights: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    values = np.asarray(values, dtype=float)
    weights = np.asarray(weights, dtype=float)
    mask = np.isfinite(values) & np.isfinite(weights) & (weights > 0)
    values = values[mask]
    weights = weights[mask]
    order = np.argsort(values)
    values = values[order]
    weights = weights[order]
    cumw = np.cumsum(weights)
    cumxw = np.cumsum(values * weights)
    relw = np.insert(cumw / cumw[-1], 0, 0.0)
    relx = np.insert(cumxw / cumxw[-1], 0, 0.0)
    return relw, relx

pass
pass

group_summary = part('figure26_equity_income_group_summary_source')
scenarios = ['Local temporal 45%', 'Spatial static 80%', 'Spatial-temporal 80%']
groups = ['Bottom 20%', '20-40%', '40-60%', '60-80%', 'Top 20%']
x = np.arange(len(groups))
pivot = group_summary.pivot(index='income_group', columns='scenario', values='mean_relative_value_exposure')

fig, ax = plt.subplots(figsize=(3, 3), dpi=180)
fig.patch.set_facecolor('white')
ax.set_facecolor('white')

for scenario in scenarios:
    vals = pivot.loc[groups, scenario].to_numpy(dtype=float) * 100
    ax.plot(
        x, vals,
        lw=2.0,
        marker='o',
        ms=4.2,
        color=COLORS[scenario],
        label=scenario
    )

ax.set_xticks(x)
ax.set_xticklabels(['Bottom\n20%', '20-40%', '40-60%', '60-80%', 'Top\n20%'], fontsize=8)
ax.set_xlim(-0.15, len(groups)-0.15)
ax.set_ylabel('Service-value exposure / income (%)', fontsize=8)

for spine in ['bottom', 'left', 'top', 'right']:
    ax.spines[spine].set_visible(True)
    ax.spines[spine].set_linewidth(0.8)
    ax.spines[spine].set_color('black')

ax.grid(True, color='#E5ECF3', lw=0.7, zorder=0, linestyle='-', alpha=0.7)
ax.set_axisbelow(True)

plt.tight_layout()
plt.show()


## Panel K

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

CWD = Path.cwd()

def locate_file(name: str) -> Path:
    direct = CWD / name
    if not direct.exists():
        raise FileNotFoundError(f"Missing current-branch input: {name}")
    return direct

DATA = INTEGRATED_DATA
df = load_integrated_table('figure26_equity_exposure_ONE_CSV.csv')

def part(name):
    return df[df['dataset'].eq(name)].copy()

COLORS = {
    'Local static 45%': '#C95E46',
    'Local temporal 45%': '#F0A44B',
    'Spatial static 80%': '#2E6E9E',
    'Spatial-temporal 80%': '#1B9A8A',
    'East': '#4C78A8',
    'Central': '#F2B84B',
    'West': '#59A14F',
    'Northeast': '#9C755F',
}

DISPLAY_LABELS = {
    'Spatial-temporal 80%': 'Spatiotemporal 80%',
}

def income_concentration(values: np.ndarray, weights: np.ndarray, income: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    values = np.asarray(values, dtype=float)
    weights = np.asarray(weights, dtype=float)
    income = np.asarray(income, dtype=float)
    mask = np.isfinite(income) & np.isfinite(values) & np.isfinite(weights) & (weights > 0)
    values = values[mask]
    weights = weights[mask]
    order = np.argsort(income[mask], kind="stable")
    values = values[order]
    weights = weights[order]
    cumw = np.cumsum(weights)
    cumxw = np.cumsum(values * weights)
    relw = np.insert(cumw / cumw[-1], 0, 0.0)
    relx = np.insert(cumxw / cumxw[-1], 0, 0.0)
    return relw, relx

pass
pass

income_exposure = part('figure26_equity_income_percentile_exposure_source')

fig, ax = plt.subplots(figsize=(3, 3), dpi=300)

for scenario in ['Local temporal 45%', 'Spatial static 80%', 'Spatial-temporal 80%']:
    sub = income_exposure[income_exposure['scenario'].eq(scenario)]
    x, y = income_concentration(
        sub['value_per_capita_cny'].to_numpy(),
        sub['cell_population'].to_numpy(),
        sub['Income_2025'].to_numpy()
    )
    ax.plot(x, y, lw=2.0, color=COLORS[scenario], label=DISPLAY_LABELS.get(scenario, scenario))

ax.plot([0, 1], [0, 1], color='#94A3B8', lw=0.9, ls='--')

ax.set_xlabel('Cumulative income-percentile population share')
ax.set_ylabel('Cumulative compute-service value exposure')

pass

ax.set_xlim(0, 1)
ax.set_ylim(0, 1)

ax.set_aspect('equal', adjustable='box')

for spine in ['bottom', 'left', 'top', 'right']:
    ax.spines[spine].set_visible(True)
    ax.spines[spine].set_linewidth(0.8)
    ax.spines[spine].set_color('black')

ax.tick_params(axis='both', which='major', labelsize=8, width=0.8, length=3.5, color='black')

ax.grid(True, color='#E5ECF3', lw=0.7)
ax.set_axisbelow(True)

ax.legend(frameon=False, fontsize=8, loc='lower right')

plt.tight_layout()
plt.show()


## Panel L

In [ ]:
from pathlib import Path
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.path import Path as MplPath
from matplotlib.patches import PathPatch, Rectangle
import numpy as np
import pandas as pd
from IPython.display import Image, display

ROOT = Path.cwd()
if not (ROOT / "data/nature_story_atlas.feather").exists():
    ROOT = Path.cwd() / "04_plotting_workspace" / "nature_story_atlas_complete_clean_20260702"
DATA = ROOT / "data/nature_story_atlas.feather"
OUT_DIR = ROOT / "figures" / "panels" / "environment_candidates"
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT = OUT_DIR / "fig3l_value_circulation_candidate.png"
OUT_SVG = OUT_DIR / "fig3l_value_circulation_candidate.svg"

data = pd.read_feather(DATA)
routes = data[data["atlas_table"].eq("f2_annual_routes")][["source_en", "target_en", "flow_twh_it"]].dropna().copy()
province_region = data[data["atlas_table"].eq("deep_province_value_carbon_water_tradeoff")][["province_en", "region_en"]].dropna().drop_duplicates()
region_map = dict(zip(province_region["province_en"], province_region["region_en"]))

region_map.setdefault("Hainan", "South China")

routes["host_region"] = routes["source_en"].map(region_map)
routes["demand_region"] = routes["target_en"].map(region_map)
routes = routes.dropna(subset=["host_region", "demand_region"]).copy()
external_total = float(data[data["atlas_table"].eq("f5_equity_concentration")]
                       .loc[lambda x: x["scenario"].eq("Spatial-temporal 80%"), "external_value_billion_cny"].iloc[0])
value_per_twh = external_total / routes["flow_twh_it"].sum()
routes["value_billion_cny"] = routes["flow_twh_it"] * value_per_twh

raw_flow = routes.groupby(["demand_region", "host_region"], as_index=False)["value_billion_cny"].sum()
left_raw = raw_flow.groupby("demand_region")["value_billion_cny"].sum()
right_raw = raw_flow.groupby("host_region")["value_billion_cny"].sum()
raw_flow["demand_region_plot"] = raw_flow["demand_region"].where(raw_flow["demand_region"].map(left_raw).ge(2.0), "Other demand")
raw_flow["host_region_plot"] = raw_flow["host_region"].where(raw_flow["host_region"].map(right_raw).ge(1.5), "Other hosting")
flow = raw_flow.groupby(["demand_region_plot", "host_region_plot"], as_index=False)["value_billion_cny"].sum()
flow = flow.rename(columns={"demand_region_plot": "demand_region", "host_region_plot": "host_region"})
flow = flow[flow["value_billion_cny"].ge(0.15)].sort_values("value_billion_cny", ascending=False).reset_index(drop=True)

left_order = ["East", "North China", "South China", "Central", "Other demand"]
right_order = ["Southwest", "Northwest", "North China", "Other hosting"]
left_totals = flow.groupby("demand_region")["value_billion_cny"].sum().reindex(left_order).dropna()
right_totals = flow.groupby("host_region")["value_billion_cny"].sum().reindex(right_order).dropna()
left_regions = left_totals.index.tolist()
right_regions = right_totals.index.tolist()

palette = {
    "East": "#D7796A",
    "North China": "#9B7BB7",
    "Northeast": "#8EA0AD",
    "Central": "#E0A12B",
    "South": "#69A88C",
    "South China": "#69A88C",
    "Southwest": "#2F9E92",
    "Northwest": "#6D8FC7",
    "Other demand": "#B8C1CC",
    "Other hosting": "#B8C1CC",
}

mpl.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans", "sans-serif"],
    "font.size": 7,
    "axes.titlesize": 8,
    "axes.labelsize": 7,
    "xtick.labelsize": 6.2,
    "ytick.labelsize": 6.2,
    "legend.fontsize": 6,
    "axes.linewidth": 0.8,
    "svg.fonttype": "none",
    "pdf.fonttype": 42,
})

def layout_nodes(totals, top=0.93, bottom=0.10, gap=0.026):
    total = float(totals.sum())
    avail = top - bottom - gap * (len(totals) - 1)
    scale = avail / total if total > 0 else 1
    y_top = top
    out = {}
    for region, val in totals.items():
        h = val * scale
        out[region] = {"top": y_top, "bottom": y_top - h, "height": h, "value": val, "cursor": y_top}
        y_top = y_top - h - gap
    return out, scale

left_nodes, scale_left = layout_nodes(left_totals)
right_nodes, scale_right = layout_nodes(right_totals)
scale = min(scale_left, scale_right)
for nodes in [left_nodes, right_nodes]:
    for region, node in nodes.items():
        node["cursor"] = node["top"]

fig, ax = plt.subplots(figsize=(3.8, 2.3), dpi=300)
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.axis("off")

x_left, x_right = 0.16, 0.84
bar_w = 0.035

for _, row in flow.iterrows():
    lnode = left_nodes[row["demand_region"]]
    rnode = right_nodes[row["host_region"]]
    h = row["value_billion_cny"] * scale
    y0_top, y0_bot = lnode["cursor"], lnode["cursor"] - h
    y1_top, y1_bot = rnode["cursor"], rnode["cursor"] - h
    lnode["cursor"] -= h
    rnode["cursor"] -= h
    verts = [
        (x_left + bar_w/2, y0_top),
        (0.43, y0_top),
        (0.57, y1_top),
        (x_right - bar_w/2, y1_top),
        (x_right - bar_w/2, y1_bot),
        (0.57, y1_bot),
        (0.43, y0_bot),
        (x_left + bar_w/2, y0_bot),
        (x_left + bar_w/2, y0_top),
    ]
    codes = [MplPath.MOVETO, MplPath.CURVE4, MplPath.CURVE4, MplPath.CURVE4,
             MplPath.LINETO, MplPath.CURVE4, MplPath.CURVE4, MplPath.CURVE4, MplPath.CLOSEPOLY]
    color = palette.get(row["host_region"], "#94A3B8")
    ax.add_patch(PathPatch(MplPath(verts, codes), facecolor=color, edgecolor="white", lw=0.25, alpha=0.62, zorder=1))

for side, nodes, x, ha in [("Demand origin", left_nodes, x_left, "right"), ("Hosting region", right_nodes, x_right, "left")]:
    for region, node in nodes.items():
        yb, yt = node["bottom"], node["top"]
        ax.add_patch(Rectangle((x - bar_w/2, yb), bar_w, yt - yb, facecolor=palette.get(region, "#94A3B8"),
                               edgecolor="white", lw=0.45, zorder=3))
        label = region.replace("North China", "North").replace("South China", "South").replace("Other demand", "Other").replace("Other hosting", "Other")
        txt_x = x - 0.025 if ha == "right" else x + 0.025
        ax.text(txt_x, (yb + yt)/2, f"{label}\n{node['value']:.1f}", ha=ha, va="center",
                fontsize=5.8, color="#26323F", linespacing=0.92)
    ax.text(x, 0.985, side, ha="center", va="top", fontsize=6.3, color="#667786", fontweight="bold")

for _, row in flow.head(3).iterrows():
    ax.text(0.50, 0.895 - 0.045 * _,
            f"{row['demand_region'].replace('North China','North').replace('South China','South').replace('Other demand','Other')} -> "
            f"{row['host_region'].replace('North China','North').replace('South China','South').replace('Other hosting','Other')}: {row['value_billion_cny']:.1f}",
            ha="center", va="center", fontsize=5.4, color="#334155")

fig.tight_layout(pad=0.35)
fig.savefig(OUT, bbox_inches="tight")
fig.savefig(OUT_SVG, bbox_inches="tight")
plt.show()

pass


## Supplementary panels

In [ ]:
from pathlib import Path
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Image, display

ROOT = Path.cwd()
if not (ROOT / "data/nature_story_atlas.feather").exists():
    ROOT = Path.cwd() / "04_plotting_workspace" / "nature_story_atlas_complete_clean_20260702"
DATA = ROOT / "data/nature_story_atlas.feather"
OUT_DIR = ROOT / "figures" / "panels" / "supplement_candidates"
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT = OUT_DIR / "supp_fig3_sensitivity_response_candidate_v3.png"
OUT_SVG = OUT_DIR / "supp_fig3_sensitivity_response_candidate_v3.svg"

data = pd.read_feather(DATA)
sens = data[data["atlas_table"].eq("f4_sensitivities")].copy()
if sens.empty:
    raise ValueError("Missing f4_sensitivities")

mpl.rcParams.update({
    "font.family": "sans-serif", "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans", "sans-serif"],
    "font.size": 7, "axes.titlesize": 8, "axes.labelsize": 7, "xtick.labelsize": 6.2, "ytick.labelsize": 6.2,
    "legend.fontsize": 6, "axes.linewidth": 0.8, "svg.fonttype": "none", "pdf.fonttype": 42,
})
GRID = "#E7ECF2"
BLUE = "#4C78A8"; RED = "#C45A52"; GREEN = "#1E9C89"; GOLD = "#D99B2B"; PURPLE = "#8E6AAE"

PANEL_LABEL_SIZE = 12

SCALE = 0.82
FIG_W, FIG_H = 6.95 * SCALE, 2.95 * SCALE

def family_group(name):
    if name.startswith("greenprice"): return "Green price"
    if name.startswith("greenaccess"): return "Green access"
    if name.startswith("routecost"): return "Route cost"
    if name.startswith("puewe"): return "PUE/WUE"
    if name.startswith("flex"): return "Flexible share"
    if name.startswith("storage_sens"): return "Storage cost"
    if name.startswith("storagecap"): return "Storage capacity"
    if name.startswith("sharecap"): return "Hosting cap"
    if name.startswith("waterstress"): return "Water-stress rule"
    return name

sens["group"] = sens["family"].map(family_group)
summary = sens.groupby("group").agg(
    carbon_min=("delta_carbon_mtco2", "min"), carbon_max=("delta_carbon_mtco2", "max"),
    cost_min=("delta_cost_billion_cny", "min"), cost_max=("delta_cost_billion_cny", "max")
).reset_index()
summary["carbon_span"] = summary[["carbon_min", "carbon_max"]].abs().max(axis=1)
summary = summary.sort_values("carbon_span", ascending=True)

color_map = {"Green price": GREEN, "Green access": GOLD, "Route cost": RED, "PUE/WUE": PURPLE,
             "Flexible share": BLUE, "Storage cost": "#6B7280", "Storage capacity": "#94A3B8",
             "Hosting cap": "#CBD5E1", "Water-stress rule": "#B8C1CC"}

fig = plt.figure(figsize=(FIG_W, FIG_H), dpi=300)
gs = fig.add_gridspec(1, 2, width_ratios=[0.95, 1.25], wspace=0.25)

ax = fig.add_subplot(gs[0, 0])
y = np.arange(len(summary))
ax.hlines(y, summary["carbon_min"], summary["carbon_max"], color="#AEB8C5", lw=3.0, alpha=0.78)
ax.scatter(summary["carbon_min"], y, color=np.where(summary["carbon_min"]<0, GREEN, RED), s=22, edgecolor="white", lw=0.4, zorder=3)
ax.scatter(summary["carbon_max"], y, color=np.where(summary["carbon_max"]<0, GREEN, RED), s=22, edgecolor="white", lw=0.4, zorder=3)
ax.axvline(0, color="#334155", lw=0.8)
ax.set_yticks(y); ax.set_yticklabels(summary["group"])
ax.set_xlabel("Change in carbon emissions (MtCO$_2$)")
ax.set_title("a", loc="left", fontweight="bold", fontsize=PANEL_LABEL_SIZE)
ax.grid(axis="x", color=GRID, lw=0.7); ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)

ax = fig.add_subplot(gs[0, 1])
for group, sub in sens.groupby("group"):
    ax.scatter(sub["delta_cost_billion_cny"], sub["delta_carbon_mtco2"], s=28,
               color=color_map.get(group,"#94A3B8"), edgecolor="white", lw=0.45, alpha=0.92, label=group)
ax.axvline(0, color="#334155", lw=0.75); ax.axhline(0, color="#334155", lw=0.75)

x_hi = sens["delta_cost_billion_cny"].max()
for group in ["Green access", "Route cost", "PUE/WUE", "Flexible share", "Storage cost"]:
    sub = sens[sens["group"].eq(group)].copy()
    if sub.empty:
        continue
    row = sub.iloc[sub[["delta_cost_billion_cny", "delta_carbon_mtco2"]].abs().sum(axis=1).argmax()]

    near_right = row["delta_cost_billion_cny"] > 0.6 * x_hi
    ax.text(row["delta_cost_billion_cny"] + (-0.12 if near_right else 0.12),
            row["delta_carbon_mtco2"] + 0.08, group, fontsize=5.6,
            color=color_map.get(group,"#334155"), ha="right" if near_right else "left", va="bottom")

ax.set_xlabel("Cost change (billion CNY)")
ax.set_ylabel("Carbon change (MtCO$_2$)")
ax.set_title("b", loc="left", fontweight="bold", fontsize=PANEL_LABEL_SIZE)
ax.grid(color=GRID, lw=0.7); ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
ax.legend(loc="upper left", bbox_to_anchor=(0.97, 1.04), frameon=False, ncol=1,
          handletextpad=0.25, labelspacing=0.35, borderaxespad=0)

fig.tight_layout(pad=0.65)
fig.savefig(OUT, bbox_inches="tight")
fig.savefig(OUT_SVG, bbox_inches="tight")
plt.show()
pass


## Supplementary panels

In [ ]:
from pathlib import Path
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Image, display

ROOT = Path.cwd()
if not (ROOT / "data/nature_story_atlas.feather").exists():
    ROOT = Path.cwd() / "04_plotting_workspace" / "nature_story_atlas_complete_clean_20260702"
DATA = ROOT / "data/nature_story_atlas.feather"
OUT_DIR = ROOT / "figures" / "panels" / "supplement_candidates"
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT = OUT_DIR / "supp_fig3_value_burden_audit_candidate.png"
OUT_SVG = OUT_DIR / "supp_fig3_value_burden_audit_candidate.svg"

data = pd.read_feather(DATA)
trade = data[data["atlas_table"].eq("deep_province_value_carbon_water_tradeoff")].copy()
stress = data[data["atlas_table"].eq("main_figure3_environment_carbon_water_stress_score_source")][
    ["province_en", "socio_weighted_water_score", "natural_weighted_water_score"]
].copy()
if trade.empty or stress.empty:
    raise ValueError("Missing value-burden source tables")

df = trade.merge(stress, on="province_en", how="left", suffixes=("", "_stress"))
df["external_value_bn"] = df["external_value_billion_cny"]
df = df[df["external_value_bn"].fillna(0).gt(0.10)].copy()
df["carbon_per_value"] = df["external_carbon_mtco2"] / df["external_value_bn"]
df["water_pressure_per_value"] = df["socio_weighted_water_score_stress"] / df["external_value_bn"]
df["region_group"] = np.where(df["region_en"].isin(["Southwest", "Northwest", "North China"]), "Resource-side hubs", "Other receivers")
shown = df.sort_values("external_value_bn", ascending=False).head(12).iloc[::-1].reset_index(drop=True)

def short_name(x):
    return "Inner M." if x == "Inner Mongolia" else x
shown["label"] = shown["province_en"].map(short_name)

mpl.rcParams.update({
    "font.family": "sans-serif", "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans", "sans-serif"],
    "font.size": 7, "axes.titlesize": 8, "axes.labelsize": 7, "xtick.labelsize": 6.2, "ytick.labelsize": 6.2,
    "legend.fontsize": 6, "axes.linewidth": 0.8, "svg.fonttype": "none", "pdf.fonttype": 42,
})
BLUE = "#6D8FC7"; TEAL = "#1E9C89"; ORANGE = "#D7796A"; RED = "#C45A52"; GOLD = "#D99B2B"; GRID = "#E7ECF2"; INK = "#26323F"
colors = np.where(shown["region_group"].eq("Resource-side hubs"), BLUE, ORANGE)
y = np.arange(len(shown))

PANEL_LABEL_SIZE = 12

SCALE = 0.82
FIG_W, FIG_H = 7.05 * SCALE, 3.72 * SCALE

fig = plt.figure(figsize=(FIG_W, FIG_H), dpi=300)
gs = fig.add_gridspec(1, 3, width_ratios=[1.0, 0.92, 1.05], wspace=0.25)

def style(ax, axis="x"):
    ax.grid(axis=axis, color=GRID, lw=0.7)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.set_axisbelow(True)

ax = fig.add_subplot(gs[0, 0])
ax.barh(y, shown["external_value_bn"], color=colors, height=0.62, alpha=0.92)
ax.set_yticks(y); ax.set_yticklabels(shown["label"])
ax.set_xlabel("External service value\n(billion CNY)")
ax.set_title("a", loc="left", fontweight="bold", fontsize=PANEL_LABEL_SIZE)
style(ax)

ax = fig.add_subplot(gs[0, 1], sharey=ax)
ax.scatter(shown["carbon_per_value"], y, s=22, color=np.where(shown["carbon_per_value"].gt(shown["carbon_per_value"].median()), RED, TEAL),
           edgecolor="white", lw=0.45, zorder=3)
for yi, val in zip(y, shown["carbon_per_value"]):
    ax.plot([0, val], [yi, yi], color="#B8C1CC", lw=0.9, alpha=0.75, zorder=1)
ax.set_yticks(y); ax.tick_params(axis="y", labelleft=False)
ax.set_xlabel("Carbon burden per value\n(MtCO$_2$ per billion CNY)")
ax.set_title("b", loc="left", fontweight="bold", fontsize=PANEL_LABEL_SIZE)
style(ax)

ax = fig.add_subplot(gs[0, 2], sharey=ax)
ax.axvline(0, color=INK, lw=0.8)
ax.scatter(shown["water_pressure_per_value"], y, s=22, color=np.where(shown["water_pressure_per_value"].gt(0), GOLD, TEAL),
           edgecolor="white", lw=0.45, zorder=3)
for yi, val in zip(y, shown["water_pressure_per_value"]):
    ax.plot([0, val], [yi, yi], color="#B8C1CC", lw=0.9, alpha=0.75, zorder=1)
ax.set_yticks(y); ax.tick_params(axis="y", labelleft=False)
ax.set_xlabel("Socio-water-pressure burden per value\n(score per billion CNY)")
ax.set_title("c", loc="left", fontweight="bold", fontsize=PANEL_LABEL_SIZE)
style(ax)

for ax_i, col, fmt, names in [
    (fig.axes[1], "carbon_per_value", "{:.2f}", ["Xinjiang", "Ningxia", "Sichuan"]),
    (fig.axes[2], "water_pressure_per_value", "{:.1f}", ["Qinghai", "Yunnan", "Inner Mongolia"]),
]:
    for _, row in shown[shown["province_en"].isin(names)].iterrows():
        ax_i.text(row[col] + 0.03 * (shown[col].max() - shown[col].min() + 1e-9), row.name,
                  fmt.format(row[col]), va="center", ha="left", fontsize=5.1, color="#667786")

fig.tight_layout(rect=[0, 0.0, 1, 1], pad=0.6)
fig.savefig(OUT, bbox_inches="tight")
fig.savefig(OUT_SVG, bbox_inches="tight")
plt.show()

pass
